# Milestone 3 - Your First Neural Network & CNNs!

In [1]:
import torch
import torch.nn as nn
import math

---
## Q1: Flattening a Tensor

**You have a batch of grayscale spectrogram images represented as a PyTorch tensor with shape (32, 1, 64, 64). You need to flatten this tensor (excluding the batch dimension) to feed it into a Fully Connected (Linear) layer. What is the size of the input features for that Linear layer?**

**Answer: 4096**

Excluding the batch dimension (32), we flatten 1 × 64 × 64 = 4096

In [2]:
tensor = torch.randn(32, 1, 64, 64)
flattened = tensor.flatten(start_dim=1)
features = 1 * 64 * 64

q1_output = {
    "original_shape": tuple(tensor.shape),
    "flattened_shape": tuple(flattened.shape),
    "linear_input_features": int(flattened.shape[1]),
    "manual_features": int(features)
}
q1_output

{'original_shape': (32, 1, 64, 64),
 'flattened_shape': (32, 4096),
 'linear_input_features': 4096,
 'manual_features': 4096}

---
## Q2: Mel-Spectrogram Shape

**You are converting a 1-second audio clip (Sample Rate = 16,000 Hz) into a Mel-Spectrogram using torchaudio. The configuration is n_fft=400, hop_length=160, n_mels=64. What is the resulting tensor shape?**

**Answer: (1, 64, 101)**

- Total samples = 16000 (1 sec × 16000 Hz)
- n_mels = 64 → frequency bins
- Time frames = floor(16000 / 160) + 1 = 100 + 1 = 101

In [3]:
import torchaudio.transforms as T

sample_rate = 16000
duration = 1
n_samples = sample_rate * duration
waveform = torch.randn(1, n_samples)

mel_spectrogram = T.MelSpectrogram(
    sample_rate=sample_rate,
    n_fft=400,
    hop_length=160,
    n_mels=64
)

mel = mel_spectrogram(waveform)
time_frames = (n_samples // 160) + 1

q2_output = {
    "waveform_shape": tuple(waveform.shape),
    "mel_shape": tuple(mel.shape),
    "expected_shape": (1, 64, int(time_frames))
}
q2_output

{'waveform_shape': (1, 16000),
 'mel_shape': (1, 64, 101),
 'expected_shape': (1, 64, 101)}

---
## Q3: Conv2d + MaxPool2d Output Shape

**Your CNN passes a (1, 64, 64) input through nn.Conv2d(..., kernel_size=3, stride=1, padding=1) followed by nn.MaxPool2d(kernel_size=2, stride=2). What is the spatial output shape (Height, Width)?**

**Answer: (32, 32)**

- Conv2d with kernel=3, stride=1, padding=1: output = (64 - 3 + 2×1)/1 + 1 = 64 → (64, 64)
- MaxPool2d with kernel=2, stride=2: output = 64/2 = 32 → (32, 32)

In [4]:
x = torch.randn(1, 1, 64, 64)
conv = nn.Conv2d(1, 16, kernel_size=3, stride=1, padding=1)
pool = nn.MaxPool2d(kernel_size=2, stride=2)

after_conv = conv(x)
after_pool = pool(after_conv)
h_after_conv = (64 - 3 + 2 * 1) // 1 + 1
h_after_pool = h_after_conv // 2

q3_output = {
    "input_shape": tuple(x.shape),
    "after_conv_shape": tuple(after_conv.shape),
    "after_pool_shape": tuple(after_pool.shape),
    "spatial_output": (int(after_pool.shape[2]), int(after_pool.shape[3])),
    "manual_spatial_output": (int(h_after_pool), int(h_after_pool))
}
q3_output

{'input_shape': (1, 1, 64, 64),
 'after_conv_shape': (1, 16, 64, 64),
 'after_pool_shape': (1, 16, 32, 32),
 'spatial_output': (32, 32),
 'manual_spatial_output': (32, 32)}

---
## Q4: Learnable Parameters in Linear Layer

**You have a hidden layer defined as nn.Linear(in_features=128, out_features=64). How many total learnable parameters (weights + biases) does this specific layer contain?**

**Answer: 8,256**

- Weights: 128 × 64 = 8,192
- Biases: 64
- Total: 8,192 + 64 = 8,256

In [5]:
linear = nn.Linear(in_features=128, out_features=64)
weight_params = linear.weight.numel()
bias_params = linear.bias.numel()
total_params = sum(p.numel() for p in linear.parameters())

q4_output = {
    "weight_shape": tuple(linear.weight.shape),
    "bias_shape": tuple(linear.bias.shape),
    "weight_params": int(weight_params),
    "bias_params": int(bias_params),
    "total_params": int(total_params),
    "manual_total": int(128 * 64 + 64)
}
q4_output

{'weight_shape': (64, 128),
 'bias_shape': (64,),
 'weight_params': 8192,
 'bias_params': 64,
 'total_params': 8256,
 'manual_total': 8256}

---
## Q5: Number of Batches in DataLoader

**Your training dataset consists of 1,050 audio files. You initialize a DataLoader with batch_size=32 and drop_last=False. How many batches will run in one epoch?**

**Answer: 33**

- With drop_last=False: ceil(1050 / 32) = ceil(32.8125) = 33
- Last batch will have 1050 - 32×32 = 1050 - 1024 = 26 samples

In [6]:
from torch.utils.data import DataLoader, TensorDataset

dataset = TensorDataset(torch.randn(1050, 1))
loader = DataLoader(dataset, batch_size=32, drop_last=False)
num_batches = len(loader)

q5_output = {
    "total_samples": 1050,
    "batch_size": 32,
    "drop_last": False,
    "num_batches": int(num_batches),
    "manual_num_batches": int(math.ceil(1050 / 32)),
    "last_batch_size": int(1050 - 32 * 32)
}
q5_output

{'total_samples': 1050,
 'batch_size': 32,
 'drop_last': False,
 'num_batches': 33,
 'manual_num_batches': 33,
 'last_batch_size': 26}

---
## Q6: CrossEntropyLoss Calculation

**You are performing a 3-class classification. The model outputs logits [2.5, 1.0, 0.1] for a target of Class 0. Using nn.CrossEntropyLoss, what is the calculated loss value (rounded to 3 decimals)?**

**Answer: 0.273**

- softmax([2.5, 1.0, 0.1]) for class 0:
  - exp(2.5) / (exp(2.5) + exp(1.0) + exp(0.1))
  - = 12.1825 / (12.1825 + 2.7183 + 1.1052) = 12.1825 / 16.006 ≈ 0.7611
- Loss = -log(0.7611) ≈ 0.273

In [7]:
logits = torch.tensor([[2.5, 1.0, 0.1]])
target = torch.tensor([0])

criterion = nn.CrossEntropyLoss()
loss = criterion(logits, target)

exp_vals = [math.exp(2.5), math.exp(1.0), math.exp(0.1)]
sum_exp = sum(exp_vals)
softmax_0 = exp_vals[0] / sum_exp
manual_loss = -math.log(softmax_0)

q6_output = {
    "logits": logits.tolist(),
    "target": int(target.item()),
    "loss": float(loss.item()),
    "loss_rounded_3": round(float(loss.item()), 3),
    "manual_softmax_class_0": float(softmax_0),
    "manual_loss": float(manual_loss)
}
q6_output

{'logits': [[2.5, 1.0, 0.10000000149011612]],
 'target': 0,
 'loss': 0.2729603350162506,
 'loss_rounded_3': 0.273,
 'manual_softmax_class_0': 0.7611229865706312,
 'manual_loss': 0.2729603223902778}

---
## Q7: SGD Weight Update

**You are training with SGD. Current Weight = 0.5, Gradient = 0.2, Learning Rate = 0.01. What is the new weight value after one optimizer step?**

**Answer: 0.498**

SGD update rule: new_weight = weight - lr × gradient
= 0.5 - 0.01 × 0.2 = 0.5 - 0.002 = 0.498

In [8]:
weight = torch.tensor([0.5], requires_grad=True)
loss = 0.2 * weight
loss.backward()

weight_before = float(weight.item())
gradient = float(weight.grad.item())

optimizer = torch.optim.SGD([weight], lr=0.01)
optimizer.step()

weight_after = float(weight.item())
manual_weight_after = float(0.5 - 0.01 * 0.2)

q7_output = {
    "weight_before": weight_before,
    "gradient": gradient,
    "weight_after": weight_after,
    "manual_weight_after": manual_weight_after
}
q7_output

{'weight_before': 0.5,
 'gradient': 0.20000000298023224,
 'weight_after': 0.49799999594688416,
 'manual_weight_after': 0.498}

---
## Q8: Padding for Same Output Size

**You want to apply a convolution with kernel_size=5 and stride=1 such that the output size remains the same as the input. What padding value must you set?**

**Answer: 2**

For same output with stride=1: padding = (kernel_size - 1) / 2 = (5 - 1) / 2 = 2

In [9]:
x = torch.randn(1, 1, 32, 32)
conv = nn.Conv2d(1, 1, kernel_size=5, stride=1, padding=2)
output = conv(x)

kernel_size = 5
padding = (kernel_size - 1) // 2

q8_output = {
    "input_shape": tuple(x.shape),
    "output_shape": tuple(output.shape),
    "same_spatial_size": bool(x.shape[2:] == output.shape[2:]),
    "padding": int(padding)
}
q8_output

{'input_shape': (1, 1, 32, 32),
 'output_shape': (1, 1, 32, 32),
 'same_spatial_size': True,
 'padding': 2}

---
## Q9: W&B Logging Command

**In a PyTorch training loop using wandb, which command is used to send the current loss and accuracy to the dashboard?**

**Answer: wandb.log({"loss": loss, "acc": accuracy})**

- `wandb.log()` is the standard method to log metrics to the W&B dashboard
- It accepts a dictionary of key-value pairs
- `wandb.save()` is for saving files, not metrics
- `wandb.track()` and `wandb.print()` don't exist

In [10]:
q9_answer = 'wandb.log({"loss": loss, "acc": accuracy})'
q9_answer

'wandb.log({"loss": loss, "acc": accuracy})'

---
## Q10: Test Accuracy Calculation

**After training, you evaluate your model on a test set of 200 spectrograms and it correctly predicts 170 of them. What is the accuracy?**

**Answer: 0.85**

Accuracy = correct / total = 170 / 200 = 0.85

In [11]:
total = 200
correct = 170
accuracy = correct / total

q10_output = {
    "total": int(total),
    "correct": int(correct),
    "accuracy": float(accuracy)
}
q10_output

{'total': 200, 'correct': 170, 'accuracy': 0.85}